In [0]:
%fs  
// execute DBFS (Databricks File System) CLI commands instead of standard Python code. It is a shortcut for running dbutils.fs.ls(...).

ls dbfs:/databricks-datasets/nyctaxi/

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/"))
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/taxizone/"))

In [0]:
spark.sql("""
          CREATE CATALOG IF NOT EXISTS nyc_taxi
          """)

In [0]:
for schema in ["bronze","silver","gold"]:
    spark.sql(f"""
              CREATE SCHEMA IF NOT EXISTS nyc_taxi.{schema}
              """)

In [0]:
from pyspark.sql import functions as F
  
yellow_trip_path = "dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/*2019*.csv.gz"
taxi_zone_path = "dbfs:/databricks-datasets/nyctaxi/taxizone/taxi_zone_lookup.csv"

raw_trips_df = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",False)
    .load(yellow_trip_path)
    .withColumn("ingestion_timestamp",F.current_timestamp())
    .withColumn("source_file",F.col("_metadata.file_path"))  
    
)

# F.col() is a PySpark function used to reference a column in a DataFrame.
# The _metadata file is a special file that is automatically created by Databricks when you read data from a file.

(
    raw_trips_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.bronze.yellow_trips_raw")
)

# .write accesses DataFrameWriter API
# Delta refers to Delta Lake, an open-source storage framework built on top of Parquet files.

taxi_zones_df = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",False)
    .load(taxi_zone_path)
    .withColumn("ingestion_timestamp",F.current_timestamp())
    .withColumn("source_file",F.col("_metadata.file_path"))
)

(
    taxi_zones_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.bronze.taxi_zones_raw")
)

print("Bronze Ingestion Complete!")

In [0]:
%sql
select count(*) from nyc_taxi.bronze.taxi_zones_raw;
select * from nyc_taxi.bronze.yellow_trips_raw limit 5;